# LangChain Tracer Base Reference

Developer-facing statements defined in `langchain_core.tracers.base`.


# `BaseTracer: _TracerCore, BaseCallbackHandler, ABC`

Abstract synchronous base class that creates, updates, completes, and persists tracing runs for chat models, text LLMs, chains, tools, and retrievers.

## Required subclass hooks

### `_persist_run`

Persists a completed root run.

```python
@abstractmethod
_persist_run(
    self,
    run: Run, # Completed root run to persist
) -> None
```

A concrete subclass must implement this method. The abstract method body does not explicitly raise `NotImplementedError`.

## Optional subclass hooks

The callback methods invoke inherited event hooks including `_on_run_create`, `_on_run_update`, `_on_chat_model_start`, `_on_llm_start`, `_on_llm_new_token`, `_on_llm_end`, `_on_llm_error`, `_on_chain_start`, `_on_chain_end`, `_on_chain_error`, `_on_tool_start`, `_on_tool_end`, `_on_tool_error`, `_on_retriever_start`, `_on_retriever_end`, and `_on_retriever_error`.

This module does not define synchronous implementations for those hooks.

## Methods

### `on_chat_model_start`

Creates and starts a chat-model run, invokes the chat-model start hook, and returns the run.

```python
on_chat_model_start(
    self,
    serialized: dict[str, Any], # Serialized model
    messages: list[list[BaseMessage]], # Message batches supplied to the model
    *,
    run_id: UUID, # Run identifier
    tags: list[str] | None = None, # Optional run tags
    parent_run_id: UUID | None = None, # Optional parent-run identifier
    metadata: dict[str, Any] | None = None, # Optional run metadata
    name: str | None = None, # Optional run name
    **kwargs: Any, # Additional run creation arguments
) -> Run # Created chat-model run
```

Chat-model completion is handled by `on_llm_end()` rather than a separate `on_chat_model_end()` callback.

### `on_llm_start`

Creates and starts a text-LLM run, invokes the LLM start hook, and returns the run.

```python
on_llm_start(
    self,
    serialized: dict[str, Any], # Serialized model
    prompts: list[str], # Prompts supplied to the model
    *,
    run_id: UUID, # Run identifier
    tags: list[str] | None = None, # Optional run tags
    parent_run_id: UUID | None = None, # Optional parent-run identifier
    metadata: dict[str, Any] | None = None, # Optional run metadata
    name: str | None = None, # Optional run name
    **kwargs: Any, # Additional run creation arguments
) -> Run # Created LLM run
```

### `on_llm_new_token`

Adds a streaming token event to an LLM or chat-model run and invokes the token hook.

```python
@override
on_llm_new_token(
    self,
    token: str | list[str | dict[str, Any]], # Token or structured content blocks
    *,
    chunk: GenerationChunk | ChatGenerationChunk | None = None, # Optional generation chunk
    run_id: UUID, # Run identifier
    parent_run_id: UUID | None = None, # Optional parent-run identifier
    **kwargs: Any, # Additional callback arguments
) -> Run # Updated run
```

### `on_retry`

Adds a retry event to an LLM run.

```python
@override
on_retry(
    self,
    retry_state: RetryCallState, # Retry state supplied by Tenacity
    *,
    run_id: UUID, # Run identifier
    **kwargs: Any, # Additional callback arguments
) -> Run # Updated run
```

### `on_llm_end`

Completes an LLM or chat-model run, ends its trace, invokes the completion hook, and returns the run.

```python
@override
on_llm_end(
    self,
    response: LLMResult, # Model response
    *,
    run_id: UUID, # Run identifier
    **kwargs: Any, # Additional callback arguments
) -> Run # Completed run
```

### `on_llm_error`

Marks an LLM run as errored, ends its trace, invokes the error hook, and returns the run.

```python
on_llm_error(
    self,
    error: BaseException, # Error raised by the model
    *,
    run_id: UUID, # Run identifier
    **kwargs: Any, # Additional callback arguments; `response` is removed when present
) -> Run # Errored run
```

### `on_chain_start`

Creates and starts a chain run, invokes the chain start hook, and returns the run.

```python
@override
on_chain_start(
    self,
    serialized: dict[str, Any], # Serialized chain
    inputs: dict[str, Any], # Chain inputs
    *,
    run_id: UUID, # Run identifier
    tags: list[str] | None = None, # Optional run tags
    parent_run_id: UUID | None = None, # Optional parent-run identifier
    metadata: dict[str, Any] | None = None, # Optional run metadata
    run_type: str | None = None, # Optional specialized run type
    name: str | None = None, # Optional run name
    **kwargs: Any, # Additional run creation arguments
) -> Run # Created chain run
```

### `on_chain_end`

Completes a chain run, ends its trace, invokes the completion hook, and returns the run.

```python
@override
on_chain_end(
    self,
    outputs: dict[str, Any], # Chain outputs
    *,
    run_id: UUID, # Run identifier
    inputs: dict[str, Any] | None = None, # Optional chain inputs
    **kwargs: Any, # Additional callback arguments
) -> Run # Completed chain run
```

### `on_chain_error`

Marks a chain run as errored, ends its trace, invokes the error hook, and returns the run.

```python
@override
on_chain_error(
    self,
    error: BaseException, # Error raised by the chain
    *,
    inputs: dict[str, Any] | None = None, # Optional chain inputs
    run_id: UUID, # Run identifier
    **kwargs: Any, # Additional callback arguments
) -> Run # Errored chain run
```

### `on_tool_start`

Creates and starts a tool run, invokes the tool start hook, and returns the run.

```python
on_tool_start(
    self,
    serialized: dict[str, Any], # Serialized tool
    input_str: str, # Tool input string
    *,
    run_id: UUID, # Run identifier
    tags: list[str] | None = None, # Optional run tags
    parent_run_id: UUID | None = None, # Optional parent-run identifier
    metadata: dict[str, Any] | None = None, # Optional run metadata
    name: str | None = None, # Optional run name
    inputs: dict[str, Any] | None = None, # Optional structured tool inputs
    **kwargs: Any, # Additional run creation arguments
) -> Run # Created tool run
```

### `on_tool_end`

Completes a tool run, ends its trace, invokes the completion hook, and returns the run.

```python
@override
on_tool_end(
    self,
    output: Any, # Tool output
    *,
    run_id: UUID, # Run identifier
    **kwargs: Any, # Additional callback arguments
) -> Run # Completed tool run
```

### `on_tool_error`

Marks a tool run as errored, ends its trace, invokes the error hook, and returns the run.

```python
@override
on_tool_error(
    self,
    error: BaseException, # Error raised by the tool
    *,
    run_id: UUID, # Run identifier
    **kwargs: Any, # Additional callback arguments
) -> Run # Errored tool run
```

### `on_retriever_start`

Creates and starts a retriever run, invokes the retriever start hook, and returns the run.

```python
on_retriever_start(
    self,
    serialized: dict[str, Any], # Serialized retriever
    query: str, # Retrieval query
    *,
    run_id: UUID, # Run identifier
    parent_run_id: UUID | None = None, # Optional parent-run identifier
    tags: list[str] | None = None, # Optional run tags
    metadata: dict[str, Any] | None = None, # Optional run metadata
    name: str | None = None, # Optional run name
    **kwargs: Any, # Additional run creation arguments
) -> Run # Created retriever run
```

### `on_retriever_error`

Marks a retriever run as errored, ends its trace, invokes the error hook, and returns the run.

```python
@override
on_retriever_error(
    self,
    error: BaseException, # Error raised by the retriever
    *,
    run_id: UUID, # Run identifier
    **kwargs: Any, # Additional callback arguments
) -> Run # Errored retriever run
```

### `on_retriever_end`

Completes a retriever run, ends its trace, invokes the completion hook, and returns the run.

```python
@override
on_retriever_end(
    self,
    documents: Sequence[Document], # Retrieved documents
    *,
    run_id: UUID, # Run identifier
    **kwargs: Any, # Additional callback arguments
) -> Run # Completed retriever run
```

## Behaviour

Starting a trace delegates to `_TracerCore._start_trace()` and then invokes `_on_run_create()`.

Ending a trace persists only runs without a parent, removes the completed run from `run_map`, decrements the tracked child count for externally injected parent runs, removes such a parent after its last child completes, and invokes `_on_run_update()`.

`copy.copy()` and `copy.deepcopy()` return the same tracer instance.

In [ ]:
import copy # Import copy to demonstrate tracer copy behaviour
from uuid import uuid4 # Import UUID generation

from langchain_core.documents import Document # Import documents for retriever output
from langchain_core.outputs import Generation, GenerationChunk, LLMResult # Import output classes
from langchain_core.tracers.base import BaseTracer # Import the tracer base class
from langchain_core.tracers.schemas import Run # Import the Run model


class MemoryTracer(BaseTracer): # Create a concrete tracer
    def __init__(self) -> None: # Initialize the tracer
        super().__init__() # Initialize the parent tracer
        self.persisted_runs: list[Run] = [] # Store completed root runs

    def _persist_run(self, run: Run) -> None: # Implement the required method
        self.persisted_runs.append(run) # Save the completed root run
        print(f"Persisted: {run.run_type} - {run.name}") # Display persistence information

    def _on_run_create(self, run: Run) -> None: # Run whenever a trace starts
        print(f"Created: {run.run_type} - {run.name}") # Display the created run

    def _on_run_update(self, run: Run) -> None: # Run whenever a trace finishes
        print(f"Updated: {run.run_type} - {run.name}") # Display the updated run

    def _on_llm_new_token( # Handle each streamed token
        self,
        run: Run, # Active LLM run
        token: str | list[str | dict], # Generated token
        chunk: GenerationChunk | None, # Optional generation chunk
    ) -> None:
        print("New token:", token) # Display the generated token


tracer = MemoryTracer() # Create the custom tracer


# -------------------- LLM trace --------------------

llm_run_id = uuid4() # Create an ID for the LLM run

tracer.on_llm_start( # Start the LLM trace
    {"name": "DemoLLM"}, # Provide serialized model information
    ["Say hello"], # Provide the prompt
    run_id=llm_run_id, # Assign the run ID
    name="Greeting LLM", # Assign a readable name
    tags=["demo"], # Add tags
    metadata={"course": "LangChain"}, # Add metadata
)

tracer.on_llm_new_token( # Record the first streamed token
    "Hello", # Provide the token
    chunk=GenerationChunk(text="Hello"), # Provide the generation chunk
    run_id=llm_run_id, # Identify the run
)

tracer.on_llm_new_token( # Record the second streamed token
    " Saad!", # Provide the token
    chunk=GenerationChunk(text=" Saad!"), # Provide the generation chunk
    run_id=llm_run_id, # Identify the run
)

llm_result = LLMResult( # Create the final LLM response
    generations=[[Generation(text="Hello Saad!")]], # Store the generated text
    llm_output={"model": "demo-model"}, # Store provider information
)

completed_llm_run = tracer.on_llm_end( # Complete the LLM trace
    llm_result, # Provide the final response
    run_id=llm_run_id, # Identify the run
)

print("\nLLM output:", completed_llm_run.outputs) # Display the stored output
print("LLM events:", [event["name"] for event in completed_llm_run.events]) # Display events


# -------------------- Chain and child tool trace --------------------

chain_run_id = uuid4() # Create an ID for the chain run

tracer.on_chain_start( # Start the chain trace
    {"name": "DemoChain"}, # Provide serialized chain information
    {"number": 5}, # Provide the chain input
    run_id=chain_run_id, # Assign the run ID
    name="Square Chain", # Assign a readable name
)

tool_run_id = uuid4() # Create an ID for the tool run

tracer.on_tool_start( # Start a child tool trace
    {"name": "SquareTool"}, # Provide serialized tool information
    "5", # Provide the tool input string
    run_id=tool_run_id, # Assign the tool run ID
    parent_run_id=chain_run_id, # Attach the tool to the chain
    name="Square Tool", # Assign a readable name
    inputs={"number": 5}, # Provide structured inputs
)

tracer.on_tool_end( # Complete the tool trace
    25, # Provide the tool output
    run_id=tool_run_id, # Identify the tool run
)

completed_chain_run = tracer.on_chain_end( # Complete the chain trace
    {"result": 25}, # Provide the chain output
    run_id=chain_run_id, # Identify the chain run
)

print("\nChain output:", completed_chain_run.outputs) # Display the chain output
print("Child run count:", len(completed_chain_run.child_runs)) # Display child count
print("Child run type:", completed_chain_run.child_runs[0].run_type) # Display child type


# -------------------- Retriever trace --------------------

retriever_run_id = uuid4() # Create an ID for the retriever run

tracer.on_retriever_start( # Start the retriever trace
    {"name": "DemoRetriever"}, # Provide serialized retriever information
    "Python", # Provide the search query
    run_id=retriever_run_id, # Assign the run ID
    name="Python Retriever", # Assign a readable name
)

documents = [ # Create retrieved documents
    Document(page_content="Python is a programming language."), # First document
    Document(page_content="Python is widely used in AI."), # Second document
]

completed_retriever_run = tracer.on_retriever_end( # Complete the retriever trace
    documents, # Provide the retrieved documents
    run_id=retriever_run_id, # Identify the retriever run
)

print(
    "\nRetrieved documents:",
    len(completed_retriever_run.outputs["documents"]),
) # Display the document count

print("Persisted root runs:", len(tracer.persisted_runs)) # Display persisted root runs
print("Active runs after completion:", len(tracer.run_map)) # Display remaining active runs

print(
    "copy.copy returns same object:",
    copy.copy(tracer) is tracer,
) # Demonstrate shallow-copy behaviour

print(
    "copy.deepcopy returns same object:",
    copy.deepcopy(tracer) is tracer,
) # Demonstrate deep-copy behaviour




---

# `AsyncBaseTracer: _TracerCore, AsyncCallbackHandler, ABC`

Abstract asynchronous base class for tracing chat-model, text-LLM, chain, tool, and retriever runs.

## Required subclass hooks

### `_persist_run`

Asynchronously persists a completed root run.

```python
@abstractmethod
@override
async _persist_run(
    self,
    run: Run, # Completed root run to persist
) -> None
```

A concrete subclass must implement this method. The abstract method body does not explicitly raise `NotImplementedError`.

## Optional subclass hooks

The following asynchronous hooks are defined as no-op methods and may be overridden:

```python
async _on_run_create(self, run: Run) -> None
async _on_run_update(self, run: Run) -> None
async _on_llm_start(self, run: Run) -> None
async _on_llm_end(self, run: Run) -> None
async _on_llm_error(self, run: Run) -> None
async _on_llm_new_token(
    self,
    run: Run,
    token: str | list[str | dict[str, Any]],
    chunk: GenerationChunk | ChatGenerationChunk | None,
) -> None
async _on_chain_start(self, run: Run) -> None
async _on_chain_end(self, run: Run) -> None
async _on_chain_error(self, run: Run) -> None
async _on_tool_start(self, run: Run) -> None
async _on_tool_end(self, run: Run) -> None
async _on_tool_error(self, run: Run) -> None
async _on_chat_model_start(self, run: Run) -> None
async _on_retriever_start(self, run: Run) -> None
async _on_retriever_end(self, run: Run) -> None
async _on_retriever_error(self, run: Run) -> None
```

## Methods

### `on_chat_model_start`

Creates a chat-model run and concurrently starts its trace and invokes its start hook.

```python
@override
async on_chat_model_start(
    self,
    serialized: dict[str, Any], # Serialized model
    messages: list[list[BaseMessage]], # Message batches supplied to the model
    *,
    run_id: UUID, # Run identifier
    parent_run_id: UUID | None = None, # Optional parent-run identifier
    tags: list[str] | None = None, # Optional run tags
    metadata: dict[str, Any] | None = None, # Optional run metadata
    name: str | None = None, # Optional run name
    **kwargs: Any, # Additional run creation arguments
) -> Any # Created chat-model run
```

### `on_llm_start`

Creates an LLM run and concurrently starts its trace and invokes its start hook.

```python
@override
async on_llm_start(
    self,
    serialized: dict[str, Any], # Serialized model
    prompts: list[str], # Prompts supplied to the model
    *,
    run_id: UUID, # Run identifier
    parent_run_id: UUID | None = None, # Optional parent-run identifier
    tags: list[str] | None = None, # Optional run tags
    metadata: dict[str, Any] | None = None, # Optional run metadata
    **kwargs: Any, # Additional run creation arguments
) -> None
```

### `on_llm_new_token`

Adds a streaming token event and awaits the token hook.

```python
@override
async on_llm_new_token(
    self,
    token: str | list[str | dict[str, Any]], # Token or structured content blocks
    *,
    chunk: GenerationChunk | ChatGenerationChunk | None = None, # Optional generation chunk
    run_id: UUID, # Run identifier
    parent_run_id: UUID | None = None, # Optional parent-run identifier
    **kwargs: Any, # Additional callback arguments
) -> None
```

### `on_retry`

Adds a retry event to an LLM run.

```python
@override
async on_retry(
    self,
    retry_state: RetryCallState, # Retry state supplied by Tenacity
    *,
    run_id: UUID, # Run identifier
    **kwargs: Any, # Additional callback arguments
) -> None
```

The method performs no awaited operation.

### `on_llm_end`

Completes an LLM or chat-model run, then concurrently invokes the completion hook and ends the trace.

```python
@override
async on_llm_end(
    self,
    response: LLMResult, # Model response
    *,
    run_id: UUID, # Run identifier
    parent_run_id: UUID | None = None, # Accepted but not used directly
    tags: list[str] | None = None, # Accepted but not used directly
    **kwargs: Any, # Additional callback arguments
) -> None
```

Chat-model completion is handled here rather than through a separate `on_chat_model_end()` callback.

### `on_llm_error`

Marks an LLM run as errored, then concurrently invokes the error hook and ends the trace.

```python
@override
async on_llm_error(
    self,
    error: BaseException, # Error raised by the model
    *,
    run_id: UUID, # Run identifier
    parent_run_id: UUID | None = None, # Accepted but not used directly
    tags: list[str] | None = None, # Accepted but not used directly
    **kwargs: Any, # Additional callback arguments
) -> None
```

### `on_chain_start`

Creates a chain run and concurrently starts its trace and invokes its start hook.

```python
@override
async on_chain_start(
    self,
    serialized: dict[str, Any], # Serialized chain
    inputs: dict[str, Any], # Chain inputs
    *,
    run_id: UUID, # Run identifier
    tags: list[str] | None = None, # Optional run tags
    parent_run_id: UUID | None = None, # Optional parent-run identifier
    metadata: dict[str, Any] | None = None, # Optional run metadata
    run_type: str | None = None, # Optional specialized run type
    name: str | None = None, # Optional run name
    **kwargs: Any, # Additional run creation arguments
) -> None
```

### `on_chain_end`

Completes a chain run, then concurrently ends its trace and invokes its completion hook.

```python
@override
async on_chain_end(
    self,
    outputs: dict[str, Any], # Chain outputs
    *,
    run_id: UUID, # Run identifier
    inputs: dict[str, Any] | None = None, # Optional chain inputs
    **kwargs: Any, # Additional callback arguments
) -> None
```

### `on_chain_error`

Marks a chain run as errored, then concurrently ends its trace and invokes its error hook.

```python
@override
async on_chain_error(
    self,
    error: BaseException, # Error raised by the chain
    *,
    inputs: dict[str, Any] | None = None, # Optional chain inputs
    run_id: UUID, # Run identifier
    **kwargs: Any, # Additional callback arguments
) -> None
```

### `on_tool_start`

Creates a tool run and concurrently starts its trace and invokes its start hook.

```python
@override
async on_tool_start(
    self,
    serialized: dict[str, Any], # Serialized tool
    input_str: str, # Tool input string
    *,
    run_id: UUID, # Run identifier
    tags: list[str] | None = None, # Optional run tags
    parent_run_id: UUID | None = None, # Optional parent-run identifier
    metadata: dict[str, Any] | None = None, # Optional run metadata
    name: str | None = None, # Accepted by the callback
    inputs: dict[str, Any] | None = None, # Optional structured tool inputs
    **kwargs: Any, # Additional callback arguments
) -> None
```

### `on_tool_end`

Completes a tool run, then concurrently ends its trace and invokes its completion hook.

```python
@override
async on_tool_end(
    self,
    output: Any, # Tool output
    *,
    run_id: UUID, # Run identifier
    **kwargs: Any, # Additional callback arguments
) -> None
```

### `on_tool_error`

Marks a tool run as errored, then concurrently ends its trace and invokes its error hook.

```python
@override
async on_tool_error(
    self,
    error: BaseException, # Error raised by the tool
    *,
    run_id: UUID, # Run identifier
    parent_run_id: UUID | None = None, # Accepted but not used directly
    tags: list[str] | None = None, # Accepted but not used directly
    **kwargs: Any, # Additional callback arguments
) -> None
```

### `on_retriever_start`

Creates a retriever run and concurrently starts its trace and invokes its start hook.

```python
@override
async on_retriever_start(
    self,
    serialized: dict[str, Any], # Serialized retriever
    query: str, # Retrieval query
    *,
    run_id: UUID, # Run identifier
    parent_run_id: UUID | None = None, # Optional parent-run identifier
    tags: list[str] | None = None, # Optional run tags
    metadata: dict[str, Any] | None = None, # Optional run metadata
    name: str | None = None, # Optional run name
    **kwargs: Any, # Additional callback arguments
) -> None
```

### `on_retriever_error`

Marks a retriever run as errored, then concurrently ends its trace and invokes its error hook.

```python
@override
async on_retriever_error(
    self,
    error: BaseException, # Error raised by the retriever
    *,
    run_id: UUID, # Run identifier
    parent_run_id: UUID | None = None, # Accepted but not used directly
    tags: list[str] | None = None, # Accepted but not used directly
    **kwargs: Any, # Additional callback arguments
) -> None
```

### `on_retriever_end`

Completes a retriever run, then concurrently ends its trace and invokes its completion hook.

```python
@override
async on_retriever_end(
    self,
    documents: Sequence[Document], # Retrieved documents
    *,
    run_id: UUID, # Run identifier
    parent_run_id: UUID | None = None, # Accepted but not used directly
    tags: list[str] | None = None, # Accepted but not used directly
    **kwargs: Any, # Additional callback arguments
) -> None
```

## Behaviour

Start callbacks use `asyncio.gather()` so trace initialization and the corresponding event hook run concurrently. End and error callbacks likewise run trace finalization and their event hooks concurrently; hooks must not depend on those concurrent lifecycle operations having completed first.

Ending a trace asynchronously persists only root runs, removes the completed run from `run_map`, manages reference counts for externally injected parent runs, and awaits `_on_run_update()`.